## Configure local inputs

Set the following environment variables to your files or folders before launching Jupyter: `MOUSEGPT_CALIBRATION_MAT`, `MOUSEGPT_SESSION_DIR`. Paths may include spaces. Output variables select destinations; input variables select existing session data. Review the analysis parameters in the cells before running on a new experiment.


In [ ]:
import os
required_paths = ['MOUSEGPT_CALIBRATION_MAT', 'MOUSEGPT_SESSION_DIR']
missing_paths = [name for name in required_paths if not os.environ.get(name)]
if missing_paths:
    raise ValueError("Set these environment variables before running: " + ", ".join(missing_paths))


In [ ]:
import os
import cv2
import json
import time
import pickle

from tqdm import tqdm
from glob import glob
from itertools import product
from collections import Counter

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from easymocap.mytools.triangulator import robust_triangulate_point, iterative_triangulate, batch_triangulate


In [ ]:
import os
# Load jsons

data_folder = os.environ['MOUSEGPT_SESSION_DIR']

jsons = []
for json_name in sorted(glob(f'{data_folder}/*.json')):
    print(json_name)
    
    with open(json_name, 'r') as f:
        raw_data = json.load(f)
        jsons.append(raw_data)

In [ ]:
# Load 2D keypoints

data_2D = []

for raw_data in jsons:
    
    data = []
    for raw_frame in tqdm(raw_data):
        
        frame = []
        for instance in raw_frame['instances']:
            kpts = np.array(instance['keypoints'])
            score = np.array(instance['keypoint_scores'])
            kpts_with_score = np.concatenate((kpts, score.reshape(len(score), -1)), axis=1)
            
            frame.append(kpts_with_score)
            
        data.append(frame)
        
    data_2D.append(data)

with open("data_2d_free_2.pkl", "wb") as f:
    pickle.dump(data_2D, f)

In [ ]:
# Statistic pose num per frame from each cam

fig = plt.figure()

for id, cam in enumerate(data_2D):
    pose_num = []
    for frame in cam:
        pose_num.append(len(frame))


    counter = Counter(pose_num)


    values = list(counter.values())
    keys = list(counter.keys())

    fig.clear()

    plt.bar(keys, values)


    plt.title('Element Counts')
    plt.xlabel('Element')
    plt.ylabel('Count')


    plt.savefig(f'cam_{id+1}_free_2.png')

In [ ]:
# Statistic cams have 2 pose in the same frame

cam_num_per_frame = []
for frame_id in range(len(data_2D[0])):
    
    num = 0
    for cam in data_2D:
        if len(cam[frame_id]) == 2:
            num += 1

    cam_num_per_frame.append(num)


counter = Counter(cam_num_per_frame)


values = list(counter.values())
keys = list(counter.keys())

fig.clear()

plt.bar(keys, values)


plt.title('Element Counts')
plt.xlabel('Element')
plt.ylabel('Count')


plt.savefig(f'cam_per_frame_free_2.png')

In [ ]:
print(counter)

In [ ]:
import os
from min_tri_mse_batch_tri import read_camera_from_mat, project

# read camera
mat_path = os.environ['MOUSEGPT_CALIBRATION_MAT']
cameras, palls, ks, rts, RDs, TDs, rs, ts = read_camera_from_mat(mat_path)

# with open("data_2d.pkl", "rb") as f:
#     data_2D = pickle.load(f)



In [ ]:
def get_match_index(select_frames):
    select_cam = list(select_frames.keys())
    select_pose = np.array(list(select_frames.values()))
    
    # Full arrangement
    indices = list(product(*[range(len(lst)) for lst in select_pose if len(lst) > 0]))
    
    all_mse_list = []
    for idxs in indices[:len(indices) // 2]:

        # tri from 2d 8view
        cams = palls[select_cam]
        res  = batch_triangulate(select_pose[range(len(select_pose)), idxs, :, :], cams)

        # cal reprojection mse
        mse = 0
        for id, view_id in enumerate(select_cam):

            # cal reproject 2d point
            res_2d = project(res[:, :3], ks[view_id], rs[view_id], ts[view_id], RDs[view_id], TDs[view_id])
            res_2d = np.squeeze(res_2d, axis=1)

            # cal mse
            mse += np.sum((res_2d - select_frames[view_id][idxs[id], :, :2]) ** 2)

        all_mse_list.append([mse, np.array(idxs)])

    all_mse_list.sort(key=lambda x:x[0])

    # print(select_cam)
    # print(select_pose.shape)
    # print(all_mse_list[0])

    mouse1 = batch_triangulate(select_pose[range(len(select_pose)), all_mse_list[0][1], :, :], cams)
    mouse2 = batch_triangulate(select_pose[range(len(select_pose)), 1-(all_mse_list[0][1]), :, :], cams)

    return [mouse1, mouse2]

In [ ]:
bones = [[ 1,  2],
        [ 2,  3],
        [ 1,  3],
        [ 1,  4],
        [ 2,  4],
        [ 3,  4],
        [ 4,  5],
        [ 7,  8],
        [ 8,  9],
        [ 9, 10],
        [11, 12],
        [12, 13],
        [13, 14],
        [14,  4],
        [15, 16],
        [16, 17],
        [17, 18],
        [18,  4],
        [19, 20],
        [20, 21],
        [22, 23],
        [23, 24],
        [ 6,  5],
        [ 6,  7],
        [ 7, 21],
        [ 7, 24]]

xmin, xmax, ymin, ymax, zmin, zmax = -0.02, 0.28, -0.02, 0.28, -0.02, 0.19

def draw_3d_kpts(ax, points, name):

    ax.scatter(points[:,0], points[:,1], points[:,2], c='r', marker='o')

    for bone in bones:
        x = [points[bone[0]-1][0], points[bone[1]-1][0]]
        y = [points[bone[0]-1][1], points[bone[1]-1][1]]
        z = [points[bone[0]-1][2], points[bone[1]-1][2]]
        ax.plot(x, y, z, c='b')

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    ax.set_xlim([xmin, xmax])
    ax.set_ylim([ymin, ymax])
    ax.set_zlim([zmin, zmax])

    ax.view_init(elev=10, azim=50)

    # plt.savefig(f'{name}.jpg')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

frame_num = len(data_2D[0])
cam_num = len(data_2D)

data_3D = []

def update(frame_id):
    print(f'frame {frame_id}', end='\r')
    
    select_frames = {}
    for cam_id in range(cam_num):
        if len(data_2D[cam_id][frame_id]) == 2:
            select_frames[cam_id] = np.array(data_2D[cam_id][frame_id])

    if len(select_frames) < 3:
        kpts_3d = np.zeros((2,24,4))
    else:
        kpts_3d = get_match_index(select_frames)

    data_3D.append(kpts_3d)
    ax.clear()

    for id, mouse in enumerate(kpts_3d):

        draw_3d_kpts(ax, mouse, f'{frame_id}-{id}')


In [ ]:
ani = FuncAnimation(fig, update, frames=range(frame_num), interval=1000/60)


ani.save('animation-all-2.mp4', writer='ffmpeg')


plt.show()

data_3D = np.array(data_3D)[..., :3]

np.save('multi_mouse_2.npy', data_3D)